# A Gentle Introduction to Scikit-Learn 🧠

**Scikit-Learn** (often imported as `sklearn`) is the most popular machine learning library in Python. It gives us ready-to-use tools to:

- Get our data ready for a model
- Pick a machine learning model (called an *estimator*)
- Train the model on our data (called *fitting*)
- Make predictions and check how good they are
- Improve the model and save it for later

---

## What we'll cover in this notebook

0. An end-to-end Scikit-Learn workflow (the big picture)
1. Getting the data ready
2. Choosing the right machine learning estimator/algorithm/model for our problem
3. Fitting our chosen model to the data and using it to make a prediction
4. Evaluating a machine learning model
5. Improving predictions through experimentation (hyperparameter tuning)
6. Saving and loading a pretrained model
7. Putting it all together in a pipeline

---

### The standard Scikit-Learn workflow

Almost every Scikit-Learn project follows these same steps:

```
1. Get the data ready (X = features, y = labels/target)
2. Split the data into training and test sets
3. Choose a model (estimator)
4. Fit the model to the training data    ->  model.fit(X_train, y_train)
5. Make predictions                       ->  model.predict(X_test)
6. Evaluate the model                     ->  model.score(X_test, y_test)
7. Improve / tune the model
8. Save the model for later use
```

> 💡 **Our example problem:** We'll use a *heart disease* dataset to predict whether a patient has heart disease (`target = 1`) or not (`target = 0`). This is a **classification** problem because we are predicting a category, not a number.

## 0. Setup — importing our tools

Before doing anything, we import the libraries we'll use:

- **`matplotlib.pyplot`** → for drawing charts and plots
- **`numpy`** → for working with numbers and arrays (fast math)
- **`pandas`** → for loading and working with tables of data (DataFrames)
- **`sklearn`** → Scikit-Learn itself, our machine learning toolkit

It's good practice to print the version of Scikit-Learn so we know exactly which version our code was written for.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sklearn
print(f"Using Scikit-Learn version: {sklearn.__version__}")

Using Scikit-Learn version: 1.9.0


## 1. Getting the data ready

The first step in any machine learning project is loading the data.

We use `pandas` to read a CSV file into a **DataFrame** (a table with rows and columns). The `.head()` method shows us the first 5 rows so we can take a quick look.

**Understanding the columns:** each row is one patient, each column is a measurement (age, cholesterol, etc.). The very last column, `target`, is what we want to predict:
- `1` → the patient **has** heart disease
- `0` → the patient **does not have** heart disease

In [2]:
heart_disease = pd.read_csv("../data/heart-disease.csv")
heart_disease.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


### Splitting the data into features (X) and labels (y)

Machine learning models learn a mapping from **features** to a **label**:

- **`X` (features)** = all the input columns we use to make a prediction. We create it by *dropping* the `target` column.
- **`y` (label / target)** = the single column we are trying to predict (`target`).

> 📌 **Convention:** in Scikit-Learn, capital `X` (the features) is usually a 2D table, and lowercase `y` (the label) is usually a 1D column. You'll see this naming everywhere in ML code.

In [ ]:
# Features (X): everything EXCEPT the target column.
# axis=1 means "drop a column" (axis=0 would drop a row).
X = heart_disease.drop("target", axis=1)

# Label (y): only the target column — this is what we want to predict.
y = heart_disease["target"]

X.head()

### Checking the balance of our labels

Before training, it's smart to check how many of each class we have. `value_counts()` counts how many `0`s and `1`s are in `y`.

If one class hugely outnumbers the other (an *imbalanced* dataset), the model can become biased. Here the two classes are fairly close (165 vs 138), which is good.

In [6]:
y.head(), y.value_counts()

(0    1
 1    1
 2    1
 3    1
 4    1
 Name: target, dtype: int64,
 target
 1    165
 0    138
 Name: count, dtype: int64)

### Splitting into training and test sets

We never test a model on the same data it learned from — that would be like giving students the exact exam questions to study. Instead, we split the data into two parts:

- **Training set** → the model *learns* from this (`X_train`, `y_train`)
- **Test set** → we *evaluate* the model on this unseen data (`X_test`, `y_test`)

`train_test_split` does this for us. `test_size=0.25` means **25% of the data is kept aside for testing** and the remaining 75% is used for training.

> 💡 **Tip:** add `random_state=42` to `train_test_split` if you want the *same* split every time you run the cell (reproducible results).

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data: 75% for training, 25% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y , test_size = 0.25)

# Check the shapes (rows, columns) of each piece to confirm the split worked.
X_train.shape, X_test.shape, y_train.shape, y_test.shape

## 2. Choosing the model (estimator) and its hyperparameters

In Scikit-Learn, a model is called an **estimator**. For this classification problem we'll use a **`RandomForestClassifier`**.

**What is a Random Forest?** Imagine asking many different decision trees to vote on the answer, then going with the majority vote. Each tree is a bit different, and combining them usually gives a more accurate, more stable result than a single tree. 🌳🌳🌳

**Hyperparameters** are the settings *we* choose before training (for example, how many trees to build). They are different from the patterns the model *learns* from the data. We can view all of a model's hyperparameters with `.get_params()`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create the model. We leave the hyperparameters at their defaults for now.
clf = RandomForestClassifier()

## 3. Fitting the model and making predictions

**Fitting** (also called *training*) is where the model learns the patterns that connect the features `X` to the label `y`. In Scikit-Learn this is always done with `.fit(X_train, y_train)`.

After fitting, we can ask the model to predict labels for new data with `.predict()`.

In [ ]:
# View all hyperparameters (and their default values) for this model.
# Notice n_estimators=100 -> the forest builds 100 trees by default.
clf.get_params()

In [ ]:
# Train the model on the TRAINING data only.
clf.fit(X=X_train, y=y_train)

### ⚠️ A common beginner mistake

The cell below will **raise an error on purpose** — it's a great learning moment!

`predict()` expects a **2D array** with the *same number of features* the model was trained on (here, 13 columns). Passing a flat list like `[0, 2, 3, 4]` gives a 1D array with only 4 values, so Scikit-Learn complains:

> `ValueError: Expected 2D array, got 1D array instead...`

**The lesson:** the data you predict on must have the same shape and columns as the data you trained on. Read the cell after this one to see correct predictions made on `X_test`.

In [ ]:
# ❌ This FAILS on purpose: a 1D array with the wrong number of features.
#    See the markdown cell above for the explanation.
y_label = clf.predict(np.array([0,2,3,4]))

In [17]:
X_train.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
103,42,1,2,120,240,1,1,194,0,0.8,0,0,3
288,57,1,0,110,335,0,1,143,1,3.0,1,1,3
272,67,1,0,120,237,0,1,71,0,1.0,1,0,2
284,61,1,0,140,207,0,0,138,1,1.9,2,1,3
144,76,0,2,140,197,0,2,116,0,1.1,1,0,2


In [18]:
X_train.shape

(227, 13)

### Making predictions the correct way

The two cells above just remind us what the training data looks like (13 columns).

Now we predict on `X_test` — data the model has **never seen**. `predict()` returns an array of `0`s and `1`s: the model's guess for each patient in the test set.

In [ ]:
# Predict the labels for the unseen test data.
y_preds = clf.predict(X=X_test)
y_preds

## 4. Evaluating the model

Predictions are only useful if they're accurate. The simplest evaluation is **accuracy**: the percentage of predictions the model got right.

`.score()` computes accuracy for us. We'll check it on **both** sets:

- **Training accuracy** — how well the model does on data it has already seen.
- **Test accuracy** — how well it does on *new* data. This is the number that really matters.

> ⚠️ **Watch for overfitting:** if the training accuracy is very high (e.g. 100%) but the test accuracy is much lower, the model has *memorized* the training data instead of learning general patterns. That gap is the sign of **overfitting**.

In [ ]:
# Accuracy on the TRAINING set (expect this to be very high).
train_acc = clf.score(X=X_train, y=y_train)
print(train_acc * 100)

In [ ]:
# Accuracy on the TEST set (the score that really matters).
test_acc = clf.score(X=X_test, y=y_test)
print(test_acc * 100)

### Looking deeper than accuracy

Accuracy alone can hide problems. Scikit-Learn gives us richer evaluation tools in `sklearn.metrics`:

- **`classification_report`** → shows *precision*, *recall* and *f1-score* for each class:
  - **Precision** = of the patients we predicted as "1", how many actually were "1"? (avoids false alarms)
  - **Recall** = of all the real "1" patients, how many did we catch? (avoids missing cases)
  - **F1-score** = a single balance of precision and recall.
- **`confusion_matrix`** → a 2×2 table of correct vs. incorrect predictions (see the next cell).
- **`accuracy_score`** → the same overall accuracy as `.score()`, just computed directly from the predictions.

In [27]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(classification_report(y_test, y_preds))

              precision    recall  f1-score   support

           0       0.83      0.74      0.78        34
           1       0.80      0.88      0.84        42

    accuracy                           0.82        76
   macro avg       0.82      0.81      0.81        76
weighted avg       0.82      0.82      0.81        76



In [ ]:
# Confusion matrix layout:
#               Predicted 0   Predicted 1
#   Actual 0  [ True Neg      False Pos  ]
#   Actual 1  [ False Neg     True Pos   ]
# The numbers on the diagonal are correct predictions.
conf_matrix = confusion_matrix(y_test, y_preds)
conf_matrix

In [30]:
acc_score = accuracy_score(y_test, y_preds)
acc_score

0.8157894736842105

## 5. Improving the model (hyperparameter tuning)

Can we make the model better? One easy experiment is to change a **hyperparameter** and see what happens.

Here we try different values of `n_estimators` (the number of trees in the forest), from 100 up to 190, and print the test accuracy for each. This is the most basic form of *hyperparameter tuning*: try several settings and keep the best.

`np.random.seed(42)` makes the randomness repeatable, so we get the same results each run.

In [31]:
np.random.seed(42)
for i in range(100,200,10):
    print(f"Trying model with {i} estimators")
    model = RandomForestClassifier(n_estimators=i).fit(X_train,y_train)
    print(f"Model accuracy on test set: {model.score(X_test, y_test)*100}")

Trying model with 100 estimators
Model accuracy on test set: 80.26315789473685
Trying model with 110 estimators
Model accuracy on test set: 78.94736842105263
Trying model with 120 estimators
Model accuracy on test set: 80.26315789473685
Trying model with 130 estimators
Model accuracy on test set: 78.94736842105263
Trying model with 140 estimators
Model accuracy on test set: 78.94736842105263
Trying model with 150 estimators
Model accuracy on test set: 78.94736842105263
Trying model with 160 estimators
Model accuracy on test set: 77.63157894736842
Trying model with 170 estimators
Model accuracy on test set: 78.94736842105263
Trying model with 180 estimators
Model accuracy on test set: 80.26315789473685
Trying model with 190 estimators
Model accuracy on test set: 81.57894736842105


### A more reliable way to compare: cross-validation

The single test-set score above can be a bit lucky or unlucky, depending on which rows ended up in the test set.

**Cross-validation** fixes this. With `cv=5`, the data is split into 5 parts ("folds"); the model is trained 5 times, each time testing on a different fold and training on the other 4. We then **average** the 5 scores. This gives a much more trustworthy estimate of how the model really performs.

In [ ]:
from sklearn.model_selection import cross_val_score

np.random.seed(42)
for i in range(100, 200, 10):
    print(f"Trying model with {i} estimators")

    # Train and get the single test-set accuracy (as before).
    model = RandomForestClassifier(n_estimators=i).fit(X_train, y_train)
    print(f"Model accuracy on test set: {model.score(X_test, y_test)*100}")

    # cross_val_score returns 5 scores (one per fold); we average them.
    cross_val = np.mean(cross_val_score(model, X, y, cv=5))
    print(f"Cross validation score: {cross_val * 100}")

## 🎉 Recap & next steps

Congratulations — you've just completed a full Scikit-Learn workflow! Here's what we did:

| Step | What we did | Key code |
|------|-------------|----------|
| 1. Get data ready | Loaded the CSV and split into `X` (features) and `y` (label) | `pd.read_csv`, `drop` |
| 2. Choose a model | Picked `RandomForestClassifier` | `RandomForestClassifier()` |
| 3. Fit & predict | Trained on the training set, predicted on the test set | `.fit()`, `.predict()` |
| 4. Evaluate | Measured accuracy, precision/recall, confusion matrix | `.score()`, `classification_report` |
| 5. Improve | Tuned `n_estimators` and used cross-validation | `cross_val_score` |

### 📌 Sections still to explore

The two final steps of a real ML project are:

**6. Saving and loading a trained model** — so you don't have to retrain every time:

```python
import joblib

# Save the trained model to a file
joblib.dump(clf, "random_forest_model.joblib")

# Load it back later
loaded_model = joblib.load("random_forest_model.joblib")
loaded_model.score(X_test, y_test)
```

**7. Putting it all together in a Pipeline** — chain data preparation and the model into one object:

```python
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("model", RandomForestClassifier())
])
pipeline.fit(X_train, y_train)
pipeline.score(X_test, y_test)
```

> 💪 **Try it yourself:** experiment with a different model (e.g. `LogisticRegression`), change `test_size`, or tune another hyperparameter and compare the cross-validation scores.